In [ ]:
from google.colab import drive
drive.mount('/content/drive') # Access Drive folder

In [ ]:
# List everything in the drive folder
!ls -la /content/drive/MyDrive/OCR_images/

In [ ]:
!wget https://ollama.com/install.sh

In [ ]:
import os

# Set the system path environment variable so Ollama can find the T4 libraries
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

!chmod +x install.sh
!sudo apt-get install zstd
!./install.sh

In [ ]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)
print("Ollama is up and running")

In [ ]:
!ollama pull qwen2.5vl
!pip install ollama
!nvidia-smi

In [ ]:
import ollama

image_path = '/content/drive/MyDrive/OCR_images/walmart-receipt.png'

messages = [
    # System configuration
    {
        'role': 'system',
        'content': "You are an OCR extraction agent. Extract data into clean JSON format. Calculate tax percentage manually."
    },
    # First Shot
    {
        'role': 'user',
        'content': "Extract data from this text found on a Target receipt, make sure the Total is correct by doing the math subtotal+tax=total and make sure the tax rate is correct by doing the math (tax_amount/subtotal)*100=calculated_tax_rate: 'TARGT STORE 102 - TOTAL 10.80 - TAX 0.80 - ITEMS: MILK 10.00'"
    },
    {
        'role': 'assistant',
        'content': '{\n  "store": "Target",\n  "subtotal": 10.00,\n  "tax_amount": 0.80,\n "total":"10.80",\n  "calculated_tax_rate": "8.0%",\n  "status": "MATCH",\n "calculated_total":"10.80"\n}'
    },
    # Error Case
    {
        'role': 'user',
        'content': "Extract data from this text found on a Coffee shop receipt, make sure the Total is correct by doing the math subtotal+tax=total and make sure the tax rate is correct by doing the math (tax_amount/subtotal)*100=calculated_tax_rate: 'CAFE LATTE 5.00 - TOTAL 6.00 - TAX 0.50'"
    },
    {
        'role': 'assistant',
        'content': '{\n  "store": "Unknown Cafe",\n  "subtotal": 5.00,\n  "tax_amount": 0.50,\n "total":"6.00",\n  "calculated_tax_rate": "10.0%",\n  "status": "ERROR_MATH_MISMATCH",\n "calculated_total":"6.00"\n}'
    },
    # Real task
    {
        'role':'user',
        'content':"Now look at this new receipt image and extract the data using the exact same JSON format.",
        'images': [image_path]
    }
]

response = ollama.chat(model='qwen2.5vl', messages=messages)
print(response['message']['content'])